# NEXUS `llm` Extension -- Real-Training Demo

Runs from `nexus_continuous/llm/` and calls the **original, full NEXUS
trainer** (`nexus_continuous.algorithms.hierarchical_ac_pqn_playground.run_training`)

Steps:
1. Bootstrap `jax` (real if installed, else the private shim in `_jax_stub/`
   -- **note:** the shim is numpy-backed and is only sufficient for the
   skill-compilation step below; real training in step 4 needs a real
   `jax`+`optax`+`mujoco_playground` install).
2. Generate a skillset with the LLM client (`mock` backend by default here
   since this sandbox has no model/API access -- swap to `backend="hf"` or
   `"openai"` for a real model; this is orthogonal to the training question).
3. Compile it with `interpreter.py` into a runnable policy module and sanity-check it.
4. Run the **real, multi-seed hand-written-vs-LLM comparison** via
   `hierarchical_ac_pqn_playground.run_training` + `policies.registry.load_policy_module`
   and plot it with `plotting.py`.

In [ ]:
import sys, os


NOTEBOOK_DIR = os.getcwd()
REPO_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from nexus_continuous.llm.jax_bootstrap import ensure_jax

used_stub = ensure_jax()
print("Using bundled jax stub (compilation-only, not enough for real training):", used_stub)

## 1. Generate a skillset with the LLM client

In [ ]:
from nexus_continuous.llm.client import LLMClient, LLMConfig, MockSkillGenerator
from nexus_continuous.llm.pipeline import generate_skillset

env_name = "CartpoleBalance"
fields = ("cart_position", "pole_angle", "cart_velocity", "pole_angular_velocity")
task_description = "Keep the pole upright and centered while minimizing oscillations."

seed = 0
client = LLMClient(LLMConfig(backend="mock", seed=seed), mock_generator=MockSkillGenerator(fields, seed=seed))
skillset = generate_skillset(
    env_name=env_name,
    observation_schema="\n".join(fields),
    task_description=task_description,
    client=client,
    allowed_fields=set(fields),
)
for s in skillset.skills:
    print(f"- {s.name}: activation_rule={s.activation_rule!r}, {len(s.reward_terms)} reward term(s)")


## 2. Compile with `interpreter.py` and sanity-check

In [ ]:
from dataclasses import asdict
import jax.numpy as jnp
from nexus_continuous.llm.interpreter import make_policy_module

policy_module = make_policy_module(asdict(skillset), field_names=fields)
print("Skills:", policy_module.SKILL_NAMES)

rng = np.random.default_rng(0)
obs = jnp.asarray(rng.standard_normal((8, len(fields))).astype("float32"))
action = jnp.asarray(rng.standard_normal((8, 1)).astype("float32"))
done = jnp.zeros((8,), dtype=bool)

rewards = policy_module.skill_rewards(obs, obs, action, None, done, None)
mask = policy_module.skill_mask(obs, None)
meta_policy = policy_module.symbolic_meta_policy(obs, None)

print("skill_rewards shape:", rewards.shape)
print("skill_mask shape:", mask.shape)
print("symbolic_meta_policy:", np.asarray(meta_policy))
print(policy_module.explain_policy())


## 3. Real multi-seed comparison: hand-written vs. LLM-generated skills

Calls the **actual NEXUS trainer** end to end -- this is exactly what
`nexus_continuous/scripts/run_llm_comparison.py` does, reproduced here so it
also lives in one notebook you can step through and plot inline. No mock
trainer is involved: `train_hand_written`/`train_llm` below call
`hierarchical_ac_pqn_playground.run_training` directly.

The training config mirrors your real `configs/cartpole_balance_nesy.yaml`
(loaded from disk if present at the standard location, else an inline copy
of that file's actual contents -- both paths use the same real config, none
of it is fabricated for this notebook).

In [ ]:
from nexus_continuous.utils import load_config

_CONFIG_PATH = os.path.join(REPO_ROOT, "configs", "cartpole_balance_nesy.yaml")

_FALLBACK_NESY_CONFIG = {
    "ALG_NAME": "nexus_ac_pqn", "ENV_NAME": "CartpoleBalance", "PLAYGROUND_IMPL": "jax",
    "POLICY": "cartpole_balance", "META_POLICY_TYPE": "nesy",
    "TOTAL_TIMESTEPS": 9830400, "NUM_ENVS": 1024, "NUM_STEPS": 64, "NUM_EPOCHS": 4,
    "NUM_MINIBATCHES": 32, "NUM_SEEDS": 1, "SEED": 0, "GAMMA": 0.99, "LAMBDA": 0.65,
    "SKILL_LAMBDA": 0.65, "META_LAMBDA": 0.80, "LR": 0.0003, "LR_START": 0.0003,
    "LR_END": 0.00005, "LR_DECAY": 1.0, "ANNEAL_LR": True, "MAX_GRAD_NORM": 1.0,
    "ACTOR_HIDDEN_SIZES": [128, 128], "CRITIC_HIDDEN_SIZES": [128, 128],
    "META_HIDDEN_SIZES": [128, 128], "NUM_CRITICS": 2, "NORM_TYPE": "layer_norm",
    "ACTIVATION": "relu", "NOISE_START": 0.30, "NOISE_FINISH": 0.02, "NOISE_DECAY": 0.80,
    "META_EPS_START": 1.0, "META_EPS_FINISH": 0.02, "META_EPS_DECAY": 0.60,
    "NORMALIZE_OBS": True, "NORMALIZE_REWARD": False, "BEHAVIOR_PENALTY_COEFF": 0.001,
    "ACTOR_UPDATE_MODE": "all_states", "PRINT_EVERY": 0, "SAVE_PATH": None,
}

if os.path.exists(_CONFIG_PATH):
    base_cfg = load_config(_CONFIG_PATH)
    print("Loaded real config from", _CONFIG_PATH)
else:
    base_cfg = dict(_FALLBACK_NESY_CONFIG)
    print("configs/cartpole_balance_nesy.yaml not found at", _CONFIG_PATH,
          "-- using the inline copy of its real contents instead.")


In [ ]:
from dataclasses import asdict as _asdict

from nexus_continuous.utils import set_global_seed
from nexus_continuous.policies.registry import load_policy_module

n_seeds = 3
OBS_FIELDS = fields

def train_hand_written(seed):
    from nexus_continuous.algorithms.hierarchical_ac_pqn_playground import run_training
    cfg = dict(base_cfg)
    cfg["ENV_NAME"] = env_name
    cfg["SEED"] = seed
    cfg["POLICY"] = "cartpole_balance"
    set_global_seed(seed)
    return run_training(cfg).metrics

def train_llm(seed):
    from nexus_continuous.algorithms.hierarchical_ac_pqn_playground import run_training
    cfg = dict(base_cfg)
    cfg["ENV_NAME"] = env_name
    cfg["SEED"] = seed
    cfg["USE_LLM_SKILLS"] = True
    cfg["LLM_SKILLSET"] = _asdict(skillset)
    cfg["OBS_FIELDS"] = OBS_FIELDS
    set_global_seed(seed)
    return run_training(cfg).metrics

hand_runs, llm_runs = [], []
training_error = None
try:
    for seed in range(n_seeds):
        hand_runs.append(train_hand_written(seed))
        llm_runs.append(train_llm(seed))
    print(f"Real training succeeded for {n_seeds} seed(s) each.")
except Exception as e:
    training_error = e